In [1]:
text = """For today’s episode, I’m handing the mic over to our books and culture lead Bri Kane. She’s here to kick off a monthly series where SciAm staffers discuss a book they all read together. We’re hoping you’ll want to join us for these virtual book clubs, not just by listening to our chats but by checking out each month’s selection for yourself.

This month the SciAm staff has been digging into the far-out world of fungus with Merlin Sheldrake’s Entangled Life.

Bri has brought along news editor Sarah Lewin Frasier, breaking news editor Adam Kovac and copy editor Jenni Hacket to discuss this month’s buddy read. Here’s their conversation.

Bri Kane: Hi, everyone. I’ve been reading Entangled Life, by Merlin Sheldrake, for weeks now, and my brain has gone all mushy with mushroom facts. I can’t stop talking about them. My friends are getting worried about the fungus in my life. So I need to talk to my favorite science nerds about all of my favorite parts about this book.

So this is Entangled Life, by Merlin Sheldrake. It’s our November buddy read pick for Scientific American: 227 pages, not including an additional 125 pages or so of notes and index in the back. The sources in this bad boy are nothing to be skipped over, I will say. This is the study of mycology, which is the study of fungus, but we’re gonna get into all the different fields of science that mycology actually branches into, which is freaky and fun.

But today on the pod, I have Sarah, news editor and reader of whatever pops up on the family’s shared Kindle account; Adam, our breaking news reporter and serious nonfiction reader; and Jeni, copy editor, who will read just about anything from physics to fairies. So hey, everyone. Can you just say hi with your name so that our listeners can, kind of, tell us all apart?
"""
tokens = list(text.encode("utf-8"))
print("text length:", len(text))
print("tokens length:", len(tokens))

text length: 1803
tokens length: 1831


In [2]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

print(merge([5, 6, 6, 7, 9, 1], (6, 7), 99))

[5, 6, 99, 9, 1]


In [3]:
vocab_size = 276
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into a new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx

merging (115, 32) into a new token 256
merging (32, 116) into a new token 257
merging (105, 110) into a new token 258
merging (101, 32) into a new token 259
merging (101, 114) into a new token 260
merging (121, 32) into a new token 261
merging (97, 110) into a new token 262
merging (44, 32) into a new token 263
merging (111, 117) into a new token 264
merging (100, 32) into a new token 265
merging (111, 110) into a new token 266
merging (116, 32) into a new token 267
merging (111, 114) into a new token 268
merging (257, 104) into a new token 269
merging (226, 128) into a new token 270
merging (270, 153) into a new token 271
merging (116, 104) into a new token 272
merging (258, 103) into a new token 273
merging (111, 32) into a new token 274
merging (101, 97) into a new token 275


In [4]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 1831
ids length: 1420
compression ratio: 1.29X


In [5]:
vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

print(decode([128]))

�


In [6]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode("hello world!"))
print(encode(""))

[104, 101, 108, 108, 274, 119, 268, 108, 100, 33]
[]


In [7]:
print(decode(encode("hello world")))

hello world


In [8]:
text2 = decode(encode(text))
print(text2 == text)

True


In [9]:
valtext = "So the thing that I can’t get out of my head is the relationship between fungus and plants and how, like, they actually trade nutrients and carbon, and like, they have this real, kind of, barter system that’s going on."
valtext2 = decode(encode(valtext))
print(valtext2 == valtext)

True


In [10]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "Hello've world123 how's are you!!!?"))

['Hello', "'ve", ' world', '123', ' how', "'s", ' are', ' you', '!!!?']
